In [36]:
from FlagEmbedding import BGEM3FlagModel, FlagReranker

In [ ]:
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True) 

In [3]:
from opensearchpy import OpenSearch
from FlagEmbedding import BGEM3FlagModel

In [15]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [18]:
INDEX_NAME = "docs-vector-index"
OPENSEARCH_HOST = "localhost"
OPENSEARCH_PORT = 9200

# 1) OpenSearch client
client = OpenSearch(
    hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_auth=("admin", os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")),
    http_compress=True,
    use_ssl=False,
    verify_certs=False,
)

In [21]:
def embed_text(text: str) -> list[float]:
    result = model.encode(
        [text],
        batch_size=1,
        max_length=512,
    )
    # BGE-M3 returns dense vectors under "dense_vecs"
    return result["dense_vecs"][0].tolist()

# 3) Create vector index
def create_index(index_name: str, dimension: int) -> None:
    if client.indices.exists(index=index_name):
        print(f"Index already exists: {index_name}")
        return

    body = {
        "settings": {
            "index": {
                "knn": True
            }
        },
        "mappings": {
            "properties": {
                "doc_id": {"type": "keyword"},
                "title": {"type": "text"},
                "content": {"type": "text"},
                "vector": {
                    "type": "knn_vector",
                    "dimension": dimension,
                    "method": {
                        "name": "hnsw",
                        "space_type": "cosinesimil",
                        "engine": "lucene"
                    }
                }
            }
        }
    }

    client.indices.create(index=index_name, body=body)
    print(f"Created index: {index_name}")

# 4) Insert one document
def insert_document(doc_id: str, title: str, content: str) -> None:
    vector = embed_text(content)

    body = {
        "doc_id": doc_id,
        "title": title,
        "content": content,
        "vector": vector
    }

    client.index(index=INDEX_NAME, id=doc_id, body=body)
    print(f"Inserted doc: {doc_id}")

# 5) Bulk insert
def bulk_insert(docs: list[dict]) -> None:
    from opensearchpy.helpers import bulk

    actions = []
    for doc in docs:
        vector = embed_text(doc["content"])
        actions.append({
            "_op_type": "index",
            "_index": INDEX_NAME,
            "_id": doc["doc_id"],
            "_source": {
                "doc_id": doc["doc_id"],
                "title": doc["title"],
                "content": doc["content"],
                "vector": vector,
            }
        })

    bulk(client, actions)
    print(f"Inserted {len(actions)} docs")

# 6) Vector search
def search_similar(query: str, k: int = 3):
    query_vector = embed_text(query)

    body = {
        "size": k,
        "query": {
            "knn": {
                "vector": {
                    "vector": query_vector,
                    "k": k
                }
            }
        }
    }

    resp = client.search(index=INDEX_NAME, body=body)
    return resp["hits"]["hits"]

In [22]:
create_index("test", dimension=1024)

Created index: test


In [ ]:
create_index(INDEX_NAME, dimension=1024)

docs = [
    {
        "doc_id": "1",
        "title": "OpenSearch란?",
        "content": "OpenSearch는 검색과 분석을 위한 오픈소스 엔진이며 벡터 검색도 지원한다."
    },
    {
        "doc_id": "2",
        "title": "RAG 설명",
        "content": "RAG는 검색 결과를 LLM 입력에 넣어 답변 정확도를 높이는 구조다."
    },
    {
        "doc_id": "3",
        "title": "벡터 검색",
        "content": "문장을 임베딩 벡터로 변환해 유사한 문서를 찾는 검색 방식이다."
    }
]

bulk_insert(docs)

Index already exists: docs-vector-index


pre tokenize: 100%|██████████| 1/1 [00:00<00:00, 3637.73it/s]


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 164.17it/s]


Inserted 3 docs


Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 176.89it/s]

0.886058 벡터 검색 문장을 임베딩 벡터로 변환해 유사한 문서를 찾는 검색 방식이다.
0.78607035 OpenSearch란? OpenSearch는 검색과 분석을 위한 오픈소스 엔진이며 벡터 검색도 지원한다.
0.7196468 RAG 설명 RAG는 검색 결과를 LLM 입력에 넣어 답변 정확도를 높이는 구조다.


In [31]:
query = "유사한 문서를 벡터로 찾는 방법"

In [32]:
results = search_similar(query, k=3)

Inference Embeddings: 100%|██████████| 1/1 [00:00<00:00, 164.14it/s]


In [33]:
for hit in results:
    print(hit["_score"], hit["_source"]["title"])

0.886058 벡터 검색
0.78607035 OpenSearch란?
0.7196468 RAG 설명


In [34]:
count = client.count(index=INDEX_NAME)
print(count["count"])

3


In [55]:
reranker = FlagReranker(
    "BAAI/bge-reranker-base",
    use_fp16=False
)

In [51]:
def rerank_documents(query: str, docs: list[dict], top_n: int = 5) -> list[dict]:
    """
    docs = [
        {"doc_id": "...", "content": "...", ...},
        ...
    ]
    """

    pairs = [[query, doc["content"]] for doc in docs]
    scores = reranker.compute_score(pairs, max_length=512)

    scored_docs = []
    for doc, score in zip(docs, scores):
        item = dict(doc)
        item["rerank_score"] = float(score)
        scored_docs.append(item)

    scored_docs.sort(key=lambda x: x["rerank_score"], reverse=True)
    return scored_docs[:top_n]

In [52]:
docs = []
for hit in results:
    src = hit["_source"]
    docs.append({
        "doc_id": src["doc_id"],
        "title": src.get("title", ""),
        "content": src["content"],
        "vector_score": hit["_score"],
    })

In [53]:
docs

[{'doc_id': '3',
  'title': '벡터 검색',
  'content': '문장을 임베딩 벡터로 변환해 유사한 문서를 찾는 검색 방식이다.',
  'vector_score': 0.886058},
 {'doc_id': '1',
  'title': 'OpenSearch란?',
  'content': 'OpenSearch는 검색과 분석을 위한 오픈소스 엔진이며 벡터 검색도 지원한다.',
  'vector_score': 0.78607035},
 {'doc_id': '2',
  'title': 'RAG 설명',
  'content': 'RAG는 검색 결과를 LLM 입력에 넣어 답변 정확도를 높이는 구조다.',
  'vector_score': 0.7196468}]

In [54]:
top_docs = rerank_documents(query, docs, top_n=5)

OutOfMemoryError: CUDA out of memory. Tried to allocate 978.00 MiB. GPU 0 has a total capacity of 23.42 GiB of which 19.81 MiB is free. Process 397558 has 20.97 GiB memory in use. Including non-PyTorch memory, this process has 2.41 GiB memory in use. Of the allocated memory 2.09 GiB is allocated by PyTorch, and 16.84 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
query = "OpenSearch에서 벡터 검색은 어떻게 하나?"
passages = [
    "OpenSearch는 knn_vector 필드를 통해 벡터 검색을 지원한다.",
    "FastAPI는 Python 웹 프레임워크다.",
    "Redis는 인메모리 데이터 저장소다."
]

pairs = [[query, p] for p in passages]

scores = reranker.compute_score(
    pairs,
    max_length=1024
)

for p, s in sorted(zip(passages, scores), key=lambda x: x[1], reverse=True):
    print(f"{s:.4f} | {p}")